# Homework 5

In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The loop keeps calling the model with a `while True` loop, then checks whether the model returned any `function_call` items.

- If there are function calls, it runs the tool, appends the tool output to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is: **no function calls in the current model response**.


In [6]:
uv add opentelemetry-api opentelemetry-sdk

/Users/betul/Desktop/llm-zoomcamp-hw5/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

### Question 1

In [2]:
from rag_helper import RAGBase
from starter import index, client

class RAGTraced(RAGBase):
    def search(self, query):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm"):
            return super().llm(prompt)

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

In [3]:
rag_traced = RAGTraced(
    index=index,
    llm_client=client,
)
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x2c23dda485a7a52572f3c7f848d43905",
        "span_id": "0xae95a6ba5e21a097",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x3885bc694aec1170",
    "start_time": "2026-07-21T20:45:13.598701Z",
    "end_time": "2026-07-21T20:45:13.720888Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "089e96c0-15b2-45b7-999a-3fc34c6ba183",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x2c23dda485a7a52572f3c7f848d43905",
        "span_id": "0x08b4ca45d2c50524",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Question 2

In [3]:
from rag_helper import RAGBase
from starter import index, client
class RAGTraced(RAGBase):
    def search(self, query):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            usage = response.usage

            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

            return response


    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

In [5]:
rag_traced = RAGTraced(
    index=index,
    llm_client=client,
)
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0xad30a8d94f7d586d575dabe54a786bfb",
        "span_id": "0xd4f34dd139fc48ab",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xae0339b038a32359",
    "start_time": "2026-07-21T20:46:49.843345Z",
    "end_time": "2026-07-21T20:46:49.863175Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "089e96c0-15b2-45b7-999a-3fc34c6ba183",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xad30a8d94f7d586d575dabe54a786bfb",
        "span_id": "0x31137efc05491ce7",
        "trace_state": "[]"
    },
    "kind": "SpanKind

Answer is 7111.

### Question 3

1.867 seconds or 1867 ms That falls in 500–2000 ms

### Question 4

In [8]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [11]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [13]:
rag_traced = RAGTraced(
    index=index,
    llm_client=client,
)
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

In [14]:
import sqlite3

conn = sqlite3.connect("traces.db")

rows = conn.execute(
    "SELECT * FROM spans"
).fetchall()

rows

[('search', 1784667199707157000, 1784667199732906000, None, None, None),
 ('llm', 1784667199737046000, 1784667206946845000, 7111, 116, None),
 ('rag', 1784667199707037000, 1784667206951296000, None, None, None)]

### Question 5

In [15]:
rows = conn.execute("""
SELECT
    name,
    COUNT(*) AS call_count,
    SUM(end_time - start_time) / 1e9 AS total_seconds,
    AVG(end_time - start_time) / 1e9 AS average_seconds
FROM spans
WHERE name != 'rag'
GROUP BY name;
""").fetchall()

rows

[('llm', 1, 7.209799, 7.209799), ('search', 1, 0.025749, 0.025749)]

In [19]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

In [20]:
rows = conn.execute("""
SELECT
    name,
    COUNT(*) AS call_count,
    SUM(end_time - start_time) / 1e9 AS total_seconds,
    AVG(end_time - start_time) / 1e9 AS average_seconds
FROM spans
WHERE name != 'rag'
GROUP BY name;
""").fetchall()

rows

[('llm', 3, 11.28153, 3.76051), ('search', 3, 0.06929, 0.023096666666666668)]

In [21]:
rows = conn.execute("""
SELECT
    name,
    (end_time - start_time) / 1e9 AS duration_seconds
FROM spans
WHERE name = 'llm'
ORDER BY start_time;
""").fetchall()

rows

[('llm', 7.209799), ('llm', 2.175934), ('llm', 1.895797)]

### Question 6

In [22]:
answer = rag_traced.rag(query)
answer = rag_traced.rag(query)

In [23]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("traces.db")

df = pd.read_sql_query(
    "SELECT * FROM spans",
    conn
)

df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784667199707157000,1784667199732906000,NaN,NaN,None
1,llm,1784667199737046000,1784667206946845000,7111.0,116.0,None
2,rag,1784667199707037000,1784667206951296000,NaN,NaN,None
3,search,1784667283857966000,1784667283879600000,NaN,NaN,None
4,llm,1784667283880736000,1784667286056670000,7111.0,117.0,None
5,rag,1784667283857910000,1784667286057497000,NaN,NaN,None
6,search,1784667345018638000,1784667345040545000,NaN,NaN,None
7,llm,1784667345042123000,1784667346937920000,7111.0,136.0,None
8,rag,1784667345018579000,1784667346939288000,NaN,NaN,None
9,search,1784667379918402000,1784667379934484000,NaN,NaN,None


They are identical.